# Test the emotion model on new images
Poetry Vision, part 2 (Renad).

Load the trained model and run it on images it has never seen (for example Shaden's scene images). These images have no emotion labels, so this is a demo of how the model behaves, not an evaluation. The real test scores are in `emotion_keras.ipynb`.

What to upload:
1. `emotion_keras_results.zip` (the zip downloaded at the end of training, it has `emotion_resnet50.keras`)
2. The images: a zip (folders inside are fine, e.g. `sea/`, `desert/`) or just pick several image files

GPU is optional here, CPU works but is slower.

In [ ]:
import os, io, glob, json, zipfile
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files

MODEL_DIR  = '/content/model'
IMAGES_DIR = '/content/test_images'
OUT        = '/content/emotion_test_results'
for d in (MODEL_DIR, IMAGES_DIR, OUT):
    os.makedirs(d, exist_ok=True)

## 1) Upload the model

In [ ]:
# Tip: the zip is big, dragging it into the Files panel on the left is often faster than the upload button
if not glob.glob(f'{MODEL_DIR}/**/*.keras', recursive=True) and os.path.exists('/content/emotion_keras_results.zip'):
    zipfile.ZipFile('/content/emotion_keras_results.zip').extractall(MODEL_DIR)

if not glob.glob(f'{MODEL_DIR}/**/*.keras', recursive=True):
    uploaded = files.upload()                       # emotion_keras_results.zip
    for name in uploaded:
        zipfile.ZipFile(name).extractall(MODEL_DIR)

model_path  = glob.glob(f'{MODEL_DIR}/**/*.keras', recursive=True)[0]
config_path = glob.glob(f'{MODEL_DIR}/**/model_config.json', recursive=True)[0]
print(model_path)

In [ ]:
model  = tf.keras.models.load_model(model_path)
config = json.load(open(config_path))

CLASSES  = config['classes']
ARABIC   = config['arabic']
IMG_SIZE = tuple(config['img_size'])
preprocess = tf.keras.applications.resnet50.preprocess_input

# Trained emotions -> the emotions used in the project
PROJECT = {'amusement': 'الفرح', 'excitement': 'الفرح', 'contentment': 'السكينة',
           'awe': 'الرهبة', 'sadness': 'الحزن', 'anger': 'الغضب', 'disgust': 'الاشمئزاز', 'fear': 'الخوف'}
print(CLASSES)

## 2) Upload the images
Pick a zip or several image files. Folders inside the zip are kept, so if the images are grouped by scene (sea, desert...) you also get a chart per scene.

In [ ]:
uploaded = files.upload()
for name, data in uploaded.items():
    if name.lower().endswith('.zip'):
        zipfile.ZipFile(io.BytesIO(data)).extractall(IMAGES_DIR)
    else:
        open(f'{IMAGES_DIR}/{name}', 'wb').write(data)

image_paths = sorted(p for p in glob.glob(f'{IMAGES_DIR}/**/*', recursive=True)
                     if p.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))
                     and '__MACOSX' not in p)
print(len(image_paths), 'images')

## 3) Predict
Same preprocessing as in training: resize to 224x224 and ResNet50 `preprocess_input`.

In [ ]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    x = tf.image.resize(np.asarray(img, dtype='float32'), IMG_SIZE)   # same resize as image_dataset_from_directory
    return img, preprocess(x)

def analyze_emotions(path, top_k=3):
    img, x = load_image(path)
    probs = model.predict(x[None], verbose=0)[0]
    order = np.argsort(probs)[::-1]

    suggestions, total = [], 0.0
    for i in order[:top_k]:
        if suggestions and (total >= 0.8 or probs[i] < 0.10):
            break
        suggestions.append({'label': CLASSES[i], 'label_ar': ARABIC[CLASSES[i]],
                            'project_emotion': PROJECT[CLASSES[i]], 'score': round(float(probs[i]), 4)})
        total += float(probs[i])

    hsv = np.asarray(img.resize((64, 64)).convert('HSV'), dtype='float32') / 255.0
    margin = float(probs[order[0]] - probs[order[1]])
    return {
        'emotion_suggestions': suggestions,
        'visual_affective_attributes': {'brightness': round(float(hsv[..., 2].mean()), 3),
                                        'saturation': round(float(hsv[..., 1].mean()), 3)},
        'uncertainty': {'top1_prob': round(float(probs[order[0]]), 4),
                        'margin_top1_top2': round(margin, 4),
                        'is_ambiguous': margin < 0.15},
        'model_version': config['model_version'],
        'probs': probs,
    }

In [ ]:
rows, all_probs = [], []
for path in image_paths:
    r = analyze_emotions(path)
    p = r['probs']
    top = np.argsort(p)[::-1]
    folder = os.path.relpath(os.path.dirname(path), IMAGES_DIR)
    rows.append({
        'image': os.path.relpath(path, IMAGES_DIR),
        'folder': '' if folder == '.' else folder,
        'emotion_1': CLASSES[top[0]], 'emotion_1_ar': ARABIC[CLASSES[top[0]]], 'prob_1': round(float(p[top[0]]), 3),
        'emotion_2': CLASSES[top[1]], 'emotion_2_ar': ARABIC[CLASSES[top[1]]], 'prob_2': round(float(p[top[1]]), 3),
        'project_emotion': PROJECT[CLASSES[top[0]]],
        'ambiguous': r['uncertainty']['is_ambiguous'],
        'brightness': r['visual_affective_attributes']['brightness'],
        'saturation': r['visual_affective_attributes']['saturation'],
    })
    all_probs.append(p)

results = pd.DataFrame(rows)
all_probs = np.array(all_probs)
results.to_csv(f'{OUT}/predictions.csv', index=False, encoding='utf-8-sig')
results.head(20)

## 4) Look at the predictions

In [ ]:
n = min(16, len(image_paths))
cols = 4
plt.figure(figsize=(16, 4.3 * int(np.ceil(n / cols))))
for k in range(n):
    r = results.iloc[k]
    plt.subplot(int(np.ceil(n / cols)), cols, k + 1)
    plt.imshow(Image.open(image_paths[k]).convert('RGB'))
    flag = '  (ambiguous)' if r['ambiguous'] else ''
    plt.title(f"{r['emotion_1']} {r['prob_1']:.2f} | {r['emotion_2']} {r['prob_2']:.2f}{flag}", fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/predictions_grid.png', dpi=120)
plt.show()

In [ ]:
# Which emotions came up overall
counts = results['emotion_1'].value_counts().reindex(CLASSES, fill_value=0)
plt.figure(figsize=(8, 4))
plt.bar(counts.index, counts.values)
plt.xticks(rotation=45); plt.ylabel('Number of images'); plt.title('Top emotion per image')
plt.tight_layout()
plt.savefig(f'{OUT}/top_emotion_counts.png', dpi=150)
plt.show()

print('Ambiguous images:', int(results['ambiguous'].sum()), 'of', len(results))

In [ ]:
# If the images were in folders (e.g. one folder per scene): average emotion per folder
if results['folder'].nunique() > 1:
    per_folder = pd.DataFrame(all_probs, columns=CLASSES).groupby(results['folder']).mean()
    plt.figure(figsize=(10, 0.5 * len(per_folder) + 2))
    plt.imshow(per_folder.values, cmap='Blues', aspect='auto')
    plt.colorbar(label='average probability')
    plt.xticks(range(len(CLASSES)), CLASSES, rotation=45)
    plt.yticks(range(len(per_folder)), per_folder.index)
    for i in range(per_folder.shape[0]):
        for j in range(per_folder.shape[1]):
            plt.text(j, i, f'{per_folder.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
    plt.title('Average emotion per folder')
    plt.tight_layout()
    plt.savefig(f'{OUT}/emotion_per_folder.png', dpi=150)
    plt.show()
    per_folder.round(3).to_csv(f'{OUT}/emotion_per_folder.csv')
else:
    print('All images are in one folder, skipping the per-folder chart.')

## 5) Full output for one image
This is exactly what the app receives from `analyze_emotions`.

In [ ]:
example = analyze_emotions(image_paths[0])
example.pop('probs')
print(image_paths[0])
print(json.dumps(example, ensure_ascii=False, indent=2))

## 6) Download the results

In [ ]:
!cd /content && rm -f emotion_test_results.zip && zip -qr emotion_test_results.zip emotion_test_results
files.download('/content/emotion_test_results.zip')